# Step 0: Imports and Reading Data
Import the cleaned data from the EDA

In [17]:
from pathlib import Path
import pandas as pd
from sklearn.neighbors import NearestNeighbors

PATH_CSV = Path("../data/processed/spotify_features_scaled.csv")

In [18]:
df = pd.read_csv(PATH_CSV)

# Step 1: Data Split
 

In [19]:
METADATA_COLS = ["id", "name", "artists"]
FEATURE_COLS = ["danceability", "energy", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

features = df[FEATURE_COLS]
metadata = df[METADATA_COLS]

print("Features shape:", features.shape)

Features shape: (1141556, 8)


# Step 2: Initialize & train the engine
We configure the K-Nearest Neighbors tape measure.
- We set `n_neighbors=31` because the algorithm will always return the seed song itself as the #1 closest match (distance of 0.0). We need 31 total to get 30 new recommendations.
- We use `metric='euclidean'` because we want the literal, straight-line physical distance between the songs in our 8-dimensional room.

In [20]:
model = NearestNeighbors(n_neighbors=101, metric='euclidean', algorithm='brute')

model.fit(features)

print("KNN Engine successfully trained and mapped")

KNN Engine successfully trained and mapped


# Step 3: The Core Recommendation Engine
Wrapping the core logic into a function.

**input:**
- Row Number / Index

**Output:**
- <= 30 closest tracks

---

## Added failsafe
if our ML model can't get a similar song to it's seed & it hasn't reached the 30 songs quota, it's best to return some of the songs rather than returning all.

The kneighbors() function returns two things: the row numbers (indices) and the literal physical distances (distances).
- A distance of 0.1 means the song is a near-perfect clone.
- A distance of 1.5 means the song shares a vibe, but is slightly different.
- A distance of 4.0+ means the algorithm is practically just guessing.

To implement your 10 ≤ x ≤ 30 rule, we simply add a maximum distance threshold. We still ask the AI for the 30 closest songs, but before we return them to the user, we slice off any song that falls outside our acceptable radius.

In [ ]:
def get_recommendations(song_index, max_distance=0.5):
    # 1. Pick out the 8 mathematical scores for our target song
    seed_song_math = features.iloc[[song_index]]
    
    # 2. Ask the AI for the closest 101 songs (1 seed + 100 recs)
    distances, indices = model.kneighbors(seed_song_math)
    
    # 3. Skip the first result (the seed song matching itself)
    recommended_indices = indices[0][1:]
    recommendation_distances = distances[0][1:]

    # 4. Filter out any songs that are further away than our max_distance threshold
    valid_mask = recommendation_distances <= max_distance
    filtered_indices = recommended_indices[valid_mask]
    filtered_distances = recommendation_distances[valid_mask]
    
    # 5. Check if we met the minimum threshold of 10 songs
    total_valid_songs = len(filtered_indices)
    
    if total_valid_songs < 10:
        # In FastAPI, this will trigger an HTTP Error to your frontend!
        raise ValueError(f"Failsafe Triggered: We only found {total_valid_songs} similar songs. We don't have enough songs to recommend.")
    
    # 6. Fetch the human-readable names for the valid songs
    recommendations = metadata.iloc[filtered_indices].copy()
    recommendations['distance_score'] = filtered_distances
    
    return recommendations

# Step 4: Execute the AI

We would test it by grabbing a random song from the dataset, and print it's results

In [22]:
seed_index = 1031455

seed_name = metadata.iloc[seed_index]['name']
seed_artist = metadata.iloc[seed_index]['artists']

print(f"Finding 30 songs mathematically similar to: '{seed_name}' by {seed_artist}")

results = get_recommendations(seed_index)
display(results)

Finding 30 songs mathematically similar to: 'ONOMATO Pairing!!!' by ['t+pazolite', 'Nanahira']


,id,name,artists,distance_score
213904,6cB3Xuej6dg7wtcwUc9U8h,Beasto Blanco,['Beasto Blanco'],0.082496
18747,2j66CeFvN9HaAavLR9XU5n,Rubber Ritual,['Robert Görl'],0.084348
992799,5Q3hChTYYix7HmG0oFVBfk,Black Hole - Original Mix,['Reborn'],0.085707
260363,5evDxMPRnkMJgDtXTdcKV4,Author Unknown,['Jack Off Jill'],0.091207
260886,0WMnEn2tqPqKOSOmgSI6AN,Output,['Cyantific'],0.093356
...,...,...,...,...
678692,7KMwKpcBNIEBRtnmsD7ALX,Breakoe,['Boaz van de Beatz'],0.149710
478102,1L6YVF1UsmSPbwhTVLjOjz,Bombshell from Hell,['Scum of the Earth'],0.149763
925788,5vZRqms671vv300WMWMe3S,No More Hollow Doors,['Crash Course In Science'],0.150011
632065,3MZxVSRLYwZmGO2hZaWZTB,Apparition,['OHMelectronic'],0.150083


# Step 5: Export the KNN Engine to be used FastAPI

In [23]:
import joblib

joblib.dump(model, 'knn_model.pkl')

print("Model successfully exported as `knn_model.pkl`")

Model successfully exported as `knn_model.pkl`
